<a href="https://colab.research.google.com/github/seeuny2022-commits/finflag/blob/main/Seeun_PARK_FinFlag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
!pip -q install google-genai datasets gradio yfinance

In [76]:
import json, time
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata, files

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Find the "flash" models your key can use (fast + free)
names = [m.name.replace("models/", "") for m in client.models.list()]
flash = [n for n in names if "flash" in n and not any(x in n for x in ["image", "tts", "live", "audio", "embed", "lite"])]
print("Flash models available:", flash)

MODEL = "gemini-3.8-flash"   # changed: gemini-2.5-flash (flash[0]) gave a 404 error
print("Using:", MODEL)

Flash models available: ['gemini-2.5-flash', 'gemini-flash-latest', 'gemini-3-flash-preview', 'gemini-3.5-flash', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-omni-1.2-flash', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']
Using: gemini-3.8-flash


In [77]:
from datasets import load_dataset

ds = load_dataset("gtfintechlab/VideoConviction")
print(ds)

df = ds[list(ds.keys())[0]].to_pandas()   # turn it into a table
print(df.shape)                            # (rows, columns)
print(df.columns.tolist())                 # the column names
df.head(3)

DatasetDict({
    train: Dataset({
        features: ['id', 'derived_inner_id', 'video_id', 'start', 'end', 'action', 'action_source', 'conviction_score', 'ticker_name', 'action_date', 'price', 'quantity', 'video_title', 'annotation_id', 'is_rec_present', 'original_inner_id', 'original_video_title', 'publishedAt', 'channelId', 'channelTitle', 'videoDescription', 'tags', 'defaultAudioLanguage', 'duration', 'isCaptionAvailable', 'viewCount', 'likeCount', 'favoriteCount', 'commentCount', 'comments', 'channelDescription', 'channelViewCount', 'channelSubscriberCount', 'videoCount', 'channelCategory', 'transcript', 'youtube_video_url', 'segment_transcript'],
        num_rows: 760
    })
})
(760, 38)
['id', 'derived_inner_id', 'video_id', 'start', 'end', 'action', 'action_source', 'conviction_score', 'ticker_name', 'action_date', 'price', 'quantity', 'video_title', 'annotation_id', 'is_rec_present', 'original_inner_id', 'original_video_title', 'publishedAt', 'channelId', 'channelTitle', 'vide

,id,derived_inner_id,video_id,start,end,action,action_source,conviction_score,ticker_name,action_date,...,commentCount,comments,channelDescription,channelViewCount,channelSubscriberCount,videoCount,channelCategory,transcript,youtube_video_url,segment_transcript
0,125899618,1,0CJU8R4oNFk,109.871177,158.538947,Buy,Title,2.0,VERI,None,...,168.0,"["".I don’t who, but someone actually needs to ...",Welcome to your chance to create the financial...,43592318,641000,1168,Category 1,"Hey Bowtie Nation, Joseph Hogue here with the...",https://www.youtube.com/watch?v=0CJU8R4oNFk,these in a minute. First up is $225 million d...
1,125899618,1,0CJU8R4oNFk,197.798669,267.774387,Buy,Selected region,2.0,ZYXI,None,...,168.0,"["".I don’t who, but someone actually needs to ...",Welcome to your chance to create the financial...,43592318,641000,1168,Category 1,"Hey Bowtie Nation, Joseph Hogue here with the...",https://www.youtube.com/watch?v=0CJU8R4oNFk,"up and make your first deposit. Zynex, ticker..."
2,125899618,1,0CJU8R4oNFk,465.293375,514.698535,Buy,Title,2.0,MITK,None,...,168.0,"["".I don’t who, but someone actually needs to ...",Welcome to your chance to create the financial...,43592318,641000,1168,Category 1,"Hey Bowtie Nation, Joseph Hogue here with the...",https://www.youtube.com/watch?v=0CJU8R4oNFk,"Our next stock, $500 million dollar Midic Sys..."


In [78]:
TEXT_COL = "segment_transcript" if "segment_transcript" in df.columns else "transcript"
VIDEO_COL = "youtube_video_url"
CHANNEL_COL = next((c for c in df.columns if "channel" in c.lower()), None)
print("text:", TEXT_COL, "| video:", VIDEO_COL, "| channel:", CHANNEL_COL)

# Remove empty or very short segments
df = df[df[TEXT_COL].fillna("").str.len() > 80].reset_index(drop=True)
df["seg_id"] = range(len(df))
print(len(df), "usable segments from", df[VIDEO_COL].nunique(), "videos")

print("\nExample segment:\n", df.loc[0, TEXT_COL][:800])

text: segment_transcript | video: youtube_video_url | channel: channelId
678 usable segments from 214 videos

Example segment:
  these in a minute. First up is $225 million dollar Veritone Inc., ticker V-E-R-I, a cloud-based AI platform that structures audio and video data. And now Nation, if that sounds like a bunch of tech jargon, just understand that Veritone is in the convergence of what's going to be the three biggest trends over the next decade. Data analysis of audio and video content, cloud-based connectivity, and an AI platform that learns to become more effective. And the potential markets for these speak for themselves. The company's AIware platform is facing markets with double-digit annual growth and tens of billions in opportunities. Revenue has jumped 244% in the last two years and was even able to increase in that tough second quarter. Analysts have price targets ranging from $15 a share to as high 


In [79]:
FLAGS = ["GUARANTEE", "URGENCY", "NO_RISK", "CERTAINTY", "PROMOTION", "ALL_IN"]

# Shuffle, then keep only ONE segment per video (so the same video can't be in both dev and test)
one_per_video = df.sample(frac=1, random_state=42).drop_duplicates(VIDEO_COL)

# Pick 70: first 20 = dev (for improving the prompt), last 50 = test (locked until the end)
sample = one_per_video.sample(min(70, len(one_per_video)), random_state=42).reset_index(drop=True)
sample["split"] = ["dev"] * 20 + ["test"] * (len(sample) - 20)

sheet = sample[["seg_id", "split", VIDEO_COL, TEXT_COL]].copy()
sheet = sheet.rename(columns={TEXT_COL: "text", VIDEO_COL: "video"})
for f in FLAGS + ["RISKY"]:
    sheet[f] = ""          # empty columns for YOU to fill in

sheet.to_csv("labeling_sheet.csv", index=False)
files.download("labeling_sheet.csv")
print(sheet.split.value_counts())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

split
test    50
dev     20
Name: count, dtype: int64


In [80]:
labels = pd.read_csv("labels.csv")      # reads the file already in Colab's file panel

FLAGS = ["GUARANTEE", "URGENCY", "NO_RISK", "CERTAINTY", "PROMOTION", "ALL_IN"]
dev = labels[labels.split == "dev"].copy()
test = labels[labels.split == "test"].copy()
print(labels.groupby("split").RISKY.agg(["count", "sum"]))

       count  sum
split            
dev       20    3
test      50    6


In [81]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# The same RISKY rule from our labeling guide
def risky_rule(row, prefix=""):
    g, u, n, c, a = (row[prefix + f] for f in ["GUARANTEE", "URGENCY", "NO_RISK", "CERTAINTY", "ALL_IN"])
    return int(g == 1 or c == 1 or a == 1 or (u == 1 and n == 1))

def report(name, y_true, y_pred):
    return {"method": name,
            "precision": round(precision_score(y_true, y_pred, zero_division=0), 2),
            "recall":    round(recall_score(y_true, y_pred, zero_division=0), 2),
            "f1":        round(f1_score(y_true, y_pred, zero_division=0), 2),
            "accuracy":  round(accuracy_score(y_true, y_pred), 2)}

# Baseline 1: always give the most common answer in dev (which is "not risky")
MOST_COMMON = int(dev.RISKY.mode()[0])
def most_common(texts):
    return [MOST_COMMON] * len(texts)

# Baseline 2: simple keyword rule. Only change this list based on DEV rows, never test.
KEYWORDS = ["guarantee", "can't lose", "free money", "easy money", "money in the bag", "10x", "100x",
            "to the moon", "don't miss", "last chance", "before it's too late", "act fast", "all in",
            "call option", "put option", "short squeeze", "no brainer", "no-brainer", "will explode"]
def keyword_rule(texts):
    return [int(any(k in t.lower() for k in KEYWORDS)) for t in texts]

pd.DataFrame([report("Baseline: most common answer", dev.RISKY, most_common(dev.text)),
              report("Baseline: keyword rule",       dev.RISKY, keyword_rule(dev.text))]).set_index("method")

,precision,recall,f1,accuracy
method,,,,
Baseline: most common answer,0.0,0.00,0.0,0.85
Baseline: keyword rule,1.0,0.67,0.8,0.95


In [82]:
PROMPT_V1 = """You are checking a transcript from a finance video for red flags, to help a teenager decide
whether it is safe to act on the advice. Judge ONLY the text below.

General rules:
- Mark a flag true only if it is clearly present. If unsure, mark false.
- The creator's own trades ("I'm buying X") count as advice.
- If the creator quotes hype in order to criticise it, or warns against hype, that is NOT a red flag.
- If the text is garbled or just one short sentence, mark everything false.

Flags:
GUARANTEE: says or strongly implies profit is certain, easy or huge ("guaranteed", "can't lose", "free money",
  "money in the bag"). Past performance, analyst targets and hedged statements ("could", "I think") do not count.
URGENCY: pushes the viewer to act quickly or warns they will miss out ("buy before Monday", "don't miss this",
  "last chance"). Patient advice ("watchlist", "wait for confirmation") does not count.
NO_RISK: recommends or endorses buying/selling AND never mentions any real risk or downside. If there is no
  recommendation at all, mark false. A boilerplate "not financial advice" alone does not count as mentioning risk.
CERTAINTY: states a confident POSITIVE prediction as fact, in a way that encourages the viewer to invest
  ("this WILL hit $500", "earnings will absolutely explode"). Hedged predictions, analyst targets, fair-value
  estimates with reasoning, warnings and predictions of a fall do not count.
PROMOTION: pushes something the creator benefits from: paid membership, Discord, course, software, sponsor,
  affiliate link or coupon code. "Like and subscribe" or free resources do not count.
ALL_IN: encourages high-risk betting as something to copy: options, margin/borrowing, leverage, short-squeeze or
  meme plays, or putting a big share of your money into one stock. Normal share buying does not count.

For every flag marked true, copy a short exact quote (max 15 words) from the transcript as evidence; otherwise use "".
Then write one sentence a 16-year-old would understand, explaining whether it is safe to act on this video.

Transcript:
\"\"\"{text}\"\"\"
"""

BATCH_SIZE = 5   # videos per request (free tier only allows ~20 requests per day)

def flag_schema():
    return {"type": "object",
            "properties": {f: {"type": "object",
                               "properties": {"present": {"type": "boolean"}, "quote": {"type": "string"}},
                               "required": ["present", "quote"]} for f in FLAGS},
            "required": FLAGS}

SCHEMA = {"type": "object",                      # one video (used by the demo app later)
          "properties": {"flags": flag_schema(), "explanation": {"type": "string"}},
          "required": ["flags", "explanation"]}

BATCH_SCHEMA = {"type": "object",                # several videos in one request
                "properties": {"results": {"type": "array", "items": {
                    "type": "object",
                    "properties": {"seg_id": {"type": "integer"}, "flags": flag_schema(),
                                   "explanation": {"type": "string"}},
                    "required": ["seg_id", "flags", "explanation"]}}},
                "required": ["results"]}

import os
CACHE = {}   # remembers answers so re-running doesn't waste your quota
if os.path.exists("gemini_results_backup.json"):      # reload saved answers, so re-running never needs the API
    with open("gemini_results_backup.json") as fh:
        for e in json.load(fh):
            CACHE[(e["prompt"], e["seg_id"])] = e["result"]
print("Loaded", len(CACHE), "saved Gemini results")

def call_gemini(contents, schema, retries=4):
    for attempt in range(retries):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=contents,
                config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json",
                                                   response_schema=schema))
            return json.loads(r.text)
        except Exception as e:
            if "PerDay" in str(e):
                print("  DAILY LIMIT reached for", MODEL, "- stop and tell Claude")
                return None
            wait = 20 * (attempt + 1)
            print(f"  error ({str(e)[:80]}), retrying in {wait}s")
            time.sleep(wait)
    return None

def analyze(text, prompt):
    # One video: used by the demo app
    return call_gemini(prompt.format(text=text), SCHEMA)

def batch_prompt(prompt, rows):
    instructions = prompt.split("Transcript:")[0]
    parts = [instructions,
             "Below are SEVERAL SEPARATE transcripts. Analyze EACH ONE INDEPENDENTLY, as if it were the only one.",
             "Quotes must come from that same transcript. Return exactly one result per transcript, with its seg_id.\n"]
    for _, r in rows.iterrows():
        parts.append(f'--- seg_id {r.seg_id} ---\n"""{r.text}"""\n')
    return "\n".join(parts)

def run(frame, prompt, prompt_name):
    todo = frame[[(prompt_name, s) not in CACHE for s in frame.seg_id]]
    for start in range(0, len(todo), BATCH_SIZE):
        chunk = todo.iloc[start:start + BATCH_SIZE]
        print(f"checking videos {start + 1}-{start + len(chunk)} of {len(todo)}...")
        res = call_gemini(batch_prompt(prompt, chunk), BATCH_SCHEMA)
        if res is None:
            print("  this batch failed; re-run the cell later to retry only the missing videos")
            continue
        for item in res["results"]:
            CACHE[(prompt_name, int(item["seg_id"]))] = item
        time.sleep(13)                      # free tier allows 5 requests per minute
    out, missing = [], 0
    for _, row in frame.iterrows():
        item = CACHE.get((prompt_name, row.seg_id))
        if item is None:
            missing += 1
            item = {"flags": {f: {"present": False, "quote": ""} for f in FLAGS}, "explanation": "ERROR"}
        rec = {"seg_id": row.seg_id, "explanation": item["explanation"]}
        for f in FLAGS:
            rec["pred_" + f] = int(item["flags"][f]["present"])
            rec["quote_" + f] = item["flags"][f]["quote"]
        out.append(rec)
    if missing:
        print(f"WARNING: {missing} videos have no result yet. Re-run this cell to retry them.")
    result = frame.merge(pd.DataFrame(out), on="seg_id")
    result["pred_RISKY"] = result.apply(lambda r: risky_rule(r, "pred_"), axis=1)
    return result

print("Batch mode ready: BATCH_SIZE =", BATCH_SIZE)

Loaded 90 saved Gemini results
Batch mode ready: BATCH_SIZE = 5


In [83]:
dev_v1 = run(dev, PROMPT_V1, "v1")

pd.DataFrame([report("Baseline: most common answer", dev.RISKY, most_common(dev.text)),
              report("Baseline: keyword rule",       dev.RISKY, keyword_rule(dev.text)),
              report("Gemini (prompt v1)",           dev_v1.RISKY, dev_v1.pred_RISKY)]).set_index("method")

,precision,recall,f1,accuracy
method,,,,
Baseline: most common answer,0.0,0.00,0.0,0.85
Baseline: keyword rule,1.0,0.67,0.8,0.95
Gemini (prompt v1),1.0,0.33,0.5,0.90


In [84]:
# Agreement per flag
display(pd.DataFrame([report(f, dev_v1[f], dev_v1["pred_" + f]) for f in FLAGS]).set_index("method"))

# Every flag disagreement, with Gemini's evidence quote
for _, r in dev_v1.iterrows():
    diffs = [f for f in FLAGS if r[f] != r["pred_" + f]]
    if diffs:
        print(f"=== seg {r.seg_id} | RISKY you: {r.RISKY} Gemini: {r.pred_RISKY}")
        for f in diffs:
            print(f"  {f}: you={r[f]} Gemini={r['pred_' + f]}  quote: \"{r['quote_' + f]}\"")
        print("  Gemini's explanation:", r.explanation, "\n")

,precision,recall,f1,accuracy
method,,,,
GUARANTEE,1.0,1.0,1.00,1.00
URGENCY,0.0,0.0,0.00,0.95
NO_RISK,0.0,0.0,0.00,0.80
CERTAINTY,0.0,0.0,0.00,0.95
PROMOTION,0.8,1.0,0.89,0.95
ALL_IN,0.0,0.0,0.00,0.95


=== seg 602 | RISKY you: 0 Gemini: 0
  NO_RISK: you=1 Gemini=0  quote: ""
  Gemini's explanation: The speaker simply shares a personal, measured opinion on MasterCard while suggesting the viewer do more research. 

=== seg 75 | RISKY you: 0 Gemini: 0
  PROMOTION: you=0 Gemini=1  quote: "thank you Genavi and Sorab for joining our Patreon. They're not gonna regret it."
  Gemini's explanation: While the hosts provide a balanced look at Oracle and mention trade risks, they plug their paid Patreon community. 

=== seg 350 | RISKY you: 0 Gemini: 0
  NO_RISK: you=0 Gemini=1  quote: "this is a stock, you know, people need to buy now."
  Gemini's explanation: The speaker promotes their private paid membership group and encourages buying the stock without explaining any downside risks. 

=== seg 555 | RISKY you: 1 Gemini: 0
  URGENCY: you=1 Gemini=0  quote: ""
  ALL_IN: you=1 Gemini=0  quote: ""
  Gemini's explanation: While the fundamental stock analysis is sensible, the video contains an exten

In [85]:
PROMPT_V2 = PROMPT_V1.replace(
    "- If the text is garbled or just one short sentence, mark everything false.",
    """- If the text is garbled or just one short sentence, mark everything false.
- Judge EACH flag separately. A video can be cautious overall and still contain one red flag.
- Any real downside counts as mentioning risk, including business problems such as rising costs,
  competition, slowing growth, debt or a high valuation.
- Generic advice like "do your own research" is NOT mentioning a risk.
- The creator's plans to buy ("I'll probably start buying under $50") count as a recommendation.
- URGENCY also counts when the pressure is to buy the creator's own product ("sign up now").
- PROMOTION needs an invitation to the viewer to join or buy. Thanking existing members does not count.
- ALL_IN includes ANY options strategy presented as something to do, including selling puts or calls.""")

PROMPT_V2 = PROMPT_V2.replace(
    "explaining whether it is safe to act on this video.",
    "describing which red flags were found, or that none were found. Do not decide safety yourself.")

print(PROMPT_V2[:2000])

You are checking a transcript from a finance video for red flags, to help a teenager decide
whether it is safe to act on the advice. Judge ONLY the text below.

General rules:
- Mark a flag true only if it is clearly present. If unsure, mark false.
- The creator's own trades ("I'm buying X") count as advice.
- If the creator quotes hype in order to criticise it, or warns against hype, that is NOT a red flag.
- If the text is garbled or just one short sentence, mark everything false.
- Judge EACH flag separately. A video can be cautious overall and still contain one red flag.
- Any real downside counts as mentioning risk, including business problems such as rising costs,
  competition, slowing growth, debt or a high valuation.
- Generic advice like "do your own research" is NOT mentioning a risk.
- The creator's plans to buy ("I'll probably start buying under $50") count as a recommendation.
- URGENCY also counts when the pressure is to buy the creator's own product ("sign up now").
- P

In [86]:
dev_v2 = run(dev, PROMPT_V2, "v2")

def flag_mistakes(res):
    return int(sum((res[f] != res["pred_" + f]).sum() for f in FLAGS))

pd.DataFrame([
    {**report("Gemini v1", dev_v1.RISKY, dev_v1.pred_RISKY), "flag_mistakes": flag_mistakes(dev_v1)},
    {**report("Gemini v2", dev_v2.RISKY, dev_v2.pred_RISKY), "flag_mistakes": flag_mistakes(dev_v2)},
]).set_index("method")

,precision,recall,f1,accuracy,flag_mistakes
method,,,,,
Gemini v1,1.0,0.33,0.5,0.90,8
Gemini v2,1.0,0.67,0.8,0.95,1


In [87]:
FINAL_PROMPT, FINAL_NAME = PROMPT_V2, "v2"     # locked: chosen on DEV, never changed after this

test_res = run(test, FINAL_PROMPT, FINAL_NAME)

results = pd.DataFrame([
    report("Baseline: most common answer", test.RISKY, most_common(test.text)),
    report("Baseline: keyword rule",       test.RISKY, keyword_rule(test.text)),
    report("Gemini (prompt v2)",           test_res.RISKY, test_res.pred_RISKY),
]).set_index("method")
print(f"Test set: {len(test)} videos, {int(test.RISKY.sum())} labelled risky by us")
results

Test set: 50 videos, 6 labelled risky by us


,precision,recall,f1,accuracy
method,,,,
Baseline: most common answer,0.0,0.00,0.00,0.88
Baseline: keyword rule,1.0,0.83,0.91,0.98
Gemini (prompt v2),1.0,0.67,0.80,0.96


In [88]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(test_res.RISKY, test_res.pred_RISKY, labels=[0, 1]).ravel()
print(f"Caught risky: {tp} | Missed risky: {fn} | False alarms: {fp} | Correctly OK: {tn}")
print("Total flag mistakes on test:", flag_mistakes(test_res), "out of", len(test_res) * len(FLAGS), "flag decisions")
pd.DataFrame([report(f, test_res[f], test_res["pred_" + f]) for f in FLAGS]).set_index("method")

Caught risky: 4 | Missed risky: 2 | False alarms: 0 | Correctly OK: 44
Total flag mistakes on test: 8 out of 300 flag decisions


,precision,recall,f1,accuracy
method,,,,
GUARANTEE,0.0,0.00,0.00,1.00
URGENCY,0.0,0.00,0.00,0.98
NO_RISK,0.8,1.00,0.89,0.98
CERTAINTY,0.0,0.00,0.00,0.96
PROMOTION,1.0,0.62,0.77,0.94
ALL_IN,1.0,0.80,0.89,0.98


In [89]:
for _, r in test_res.iterrows():
    diffs = [f for f in FLAGS if r[f] != r["pred_" + f]]
    if diffs or r.RISKY != r.pred_RISKY:
        kind = "MISSED RISKY" if (r.RISKY == 1 and r.pred_RISKY == 0) else \
               "FALSE ALARM" if (r.RISKY == 0 and r.pred_RISKY == 1) else "flag mismatch only"
        print(f"=== seg {r.seg_id} | {kind} | RISKY you: {r.RISKY} Gemini: {r.pred_RISKY}")
        for f in diffs:
            print(f"  {f}: you={r[f]} Gemini={r['pred_' + f]}  quote: \"{r['quote_' + f]}\"")
        print("  Gemini:", r.explanation, "\n")

=== seg 45 | flag mismatch only | RISKY you: 0 Gemini: 0
  PROMOTION: you=1 Gemini=0  quote: ""
  Gemini: No red flags were found in this video. 

=== seg 600 | flag mismatch only | RISKY you: 0 Gemini: 0
  PROMOTION: you=1 Gemini=0  quote: ""
  Gemini: No red flags were found; the hosts discuss Google's valuation and warn that it is currently overpriced, advising patience. 

=== seg 184 | flag mismatch only | RISKY you: 0 Gemini: 0
  URGENCY: you=1 Gemini=0  quote: ""
  Gemini: No red flags were found, as the creator focuses heavily on risk management and explains why they sold their position due to fraud allegations. 

=== seg 165 | flag mismatch only | RISKY you: 1 Gemini: 1
  PROMOTION: you=1 Gemini=0  quote: ""
  Gemini: This video features an all-in red flag because it showcases a risky options trading strategy as something to do. 

=== seg 245 | MISSED RISKY | RISKY you: 1 Gemini: 0
  CERTAINTY: you=1 Gemini=0  quote: ""
  Gemini: The speaker recommends buying Arista Networks fo

In [90]:
# Fairness check: does Gemini disagree with us more for some creators than others?
both = pd.concat([dev_v2, test_res], ignore_index=True)
both["channel"] = both.seg_id.map(df.set_index("seg_id")[CHANNEL_COL]) if CHANNEL_COL else "unknown"
both["mistakes"] = sum((both[f] != both["pred_" + f]).astype(int) for f in FLAGS)
fair = both.groupby("channel").agg(videos=("seg_id", "size"),
                                   our_risky_rate=("RISKY", "mean"),
                                   gemini_risky_rate=("pred_RISKY", "mean"),
                                   flag_mistakes_per_video=("mistakes", "mean"))
fair = fair[fair.videos >= 3].sort_values("flag_mistakes_per_video", ascending=False).round(2)
print("Channels with 3+ videos in our labelled set:")
fair

Channels with 3+ videos in our labelled set:


,videos,our_risky_rate,gemini_risky_rate,flag_mistakes_per_video
channel,,,,
UC0BGhWsIbV7Dm-lsvhdlMbA,5,0.40,0.00,0.40
UChBVf9YnourrEDTsbbwJPRA,5,0.20,0.20,0.40
UCsKdUbXfgsrxtGdpaPoOiFQ,5,0.20,0.20,0.40
UCJtfma0mE_XrBAD9uakcjfA,3,0.33,0.33,0.33
UCbKdotYtcY9SxoU8CYAXdvg,8,0.12,0.00,0.12
UCoMzWLaPjDJBbipihD694pQ,9,0.22,0.22,0.11
UCSglJMvX-zSgv3PEJIE_inw,10,0.00,0.00,0.00
UC4SK8IQ_u14VjUE8KtugSmw,5,0.00,0.00,0.00
UChub1tZZuWn9YYHJZZYTbMg,3,0.00,0.00,0.00


In [91]:
results.to_csv("test_results.csv")
test_res.drop(columns=["text"]).to_csv("test_predictions.csv", index=False)
files.download("test_results.csv")
files.download("test_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [92]:
import gradio as gr

NAMES = {"GUARANTEE": "Promises guaranteed or huge returns", "URGENCY": "Pressures you to act now",
         "NO_RISK": "Recommends without mentioning any risk", "CERTAINTY": "States predictions as facts",
         "PROMOTION": "Pushes a paid group, course or product", "ALL_IN": "Encourages risky bets (options, leverage...)"}

# Results we already have (no new Gemini requests)
SAVED = {r.text.strip(): CACHE.get((FINAL_NAME, r.seg_id)) for _, r in pd.concat([dev, test]).iterrows()}

def check(transcript):
    t = transcript.strip()
    if len(t) < 40:
        return "Please paste a longer transcript."
    res = SAVED.get(t) or analyze(t, FINAL_PROMPT)   # only calls Gemini for NEW text
    if res is None:
        return "Gemini is busy or out of free requests right now. Try one of the examples below."
    flags = {f: int(res["flags"][f]["present"]) for f in FLAGS}
    head = "## 🚩 RISKY: think twice before acting on this" if risky_rule(flags) else "## ✅ No major red flags found"
    lines = [head, res["explanation"], ""]
    for f in FLAGS:
        if flags[f]:
            lines.append(f"- **{NAMES[f]}**: \"{res['flags'][f]['quote']}\"")
    lines.append("\n*FinFlag is a prototype, not financial advice. It can miss red flags: see our test results.*")
    return "\n".join(lines)

short = test_res.assign(n=test_res.text.str.len()).sort_values("n")
examples = []
for group in [short[short.pred_RISKY == 1],                                   # one flagged as risky
              short[(short.pred_RISKY == 0) & (short.pred_PROMOTION == 0)],   # one clean
              short[short.pred_PROMOTION == 1]]:                              # one with a promotion
    if len(group):
        examples.append(group.text.iloc[len(group) // 2])

gr.Interface(fn=check,
             inputs=gr.Textbox(lines=12, label="Paste a finance video transcript"),
             outputs=gr.Markdown(),
             title="FinFlag: check a finance video before you act on it",
             description="Finds red flags (guaranteed returns, pressure, hidden promotion, risky bets) and quotes the exact words.",
             examples=[[e] for e in examples]).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c93d6a8437a4678132.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [93]:
# Save every Gemini answer (v1 + v2 on dev, final test) so results can be checked without calling the API
saved = [{"prompt": k[0], "seg_id": int(k[1]), "result": v} for k, v in CACHE.items() if v is not None]
if len(saved) < 90:
    print("NOT saving: only", len(saved), "results in memory (expected 90). Your saved file is unchanged.")
else:
    with open("gemini_results_backup.json", "w") as fh:
        json.dump(saved, fh)
    print(len(saved), "results saved")
    files.download("gemini_results_backup.json")

90 results saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [94]:
# All 20 answers from AI Studio (Gemini 3.8 Flash)
AI_STUDIO_RESULTS = [
    # batch 1
    {"seg_id": 60, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video."},
    {"seg_id": 165, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": True, "quote": "And that's the beauty of this particular strategy."}}, "explanation": "This video features an all-in red flag because it showcases a risky options trading strategy as something to do."},
    {"seg_id": 245, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": True, "quote": "Arista Networks is actually one of my favorite growth stocks"}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "The speaker recommends buying Arista Networks for your portfolio without mentioning any risks or downsides."},
    {"seg_id": 265, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": True, "quote": "Our first cheap value stock to buy is Campbell Soup."}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "The video recommends buying Campbell Soup stock but fails to mention any risks or negative factors."},
    {"seg_id": 18, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video."},
    # batch 2
    {"seg_id": 349, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 163, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 557, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found because the transcript text is garbled."},
    {"seg_id": 443, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": True, "quote": "which I almost feel bad about touting in yet another video"}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "The creator endorses NVIDIA but does not mention any risks or downsides to investing in the company."},
    {"seg_id": 648, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    # batch 3
    {"seg_id": 653, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": True, "quote": "leave a link in the description below to our Patreon page"}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "The video includes a promotion inviting viewers to join the creator's paid Patreon page and private Discord community."},
    {"seg_id": 244, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 57, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 344, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 256, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": True, "quote": "this is a technical setup and short squeeze play"}}, "explanation": "The video contains an all-in red flag by encouraging viewers to trade a high-risk short squeeze and meme stock."},
    # batch 4
    {"seg_id": 223, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 283, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": True, "quote": "I'm gonna buy in a little bit this week"}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "The creator talks about buying into the investment to capture an 8% gain without mentioning any risks or downsides."},
    {"seg_id": 532, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 666, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": False, "quote": ""}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "No red flags were found in this video clip."},
    {"seg_id": 462, "flags": {"GUARANTEE": {"present": False, "quote": ""}, "URGENCY": {"present": False, "quote": ""}, "NO_RISK": {"present": True, "quote": "I keep putting more and more in there."}, "CERTAINTY": {"present": False, "quote": ""}, "PROMOTION": {"present": False, "quote": ""}, "ALL_IN": {"present": False, "quote": ""}}, "explanation": "The creator endorses buying more CVS stock without mentioning any potential risks or downsides."},
]

added = []
for item in AI_STUDIO_RESULTS:
    sid = int(item["seg_id"])
    if sid not in set(test.seg_id):
        print("skipped unexpected seg_id", sid)
        continue
    CACHE[(FINAL_NAME, sid)] = item
    added.append(sid)

still = [s for s in test.seg_id if (FINAL_NAME, s) not in CACHE]
print("Added:", len(added), "videos")
print("Test videos still missing:", len(still), still)

Added: 20 videos
Test videos still missing: 0 []
